In [3]:
import torch
import pandas as pd
import numpy as np
import regex as re
from epsilon_transformers.persistence import Persister


/opt/anaconda3/envs/epstrans311/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/opt/anaconda3/envs/epstrans311/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because a

In [4]:
persister = Persister(save_dir="/Users/sbhandari/Documents/GitHub/epsilon-transformers/models/1layer_0.15_0.6_lr0.02_2000_thry2/epsilon-transformers/models/linear_mess3_thry2/1layer_0.15_0.6_lr0.02_2000")
# model_path="/Users/sbhandari/Documents/GitHub/epsilon-transformers/models/small checkpoints/1lyr_0.15_0.6_lr0.01adm_40M_400/epsilon-transformers/models/linrmess3_thry/1lyr_0.15_0.6_lr0.01adm_40M_400/checkpoint_801_tokens_40000000.pt"
model = persister.load_final_model()

[Persister] Found 2001 checkpoints in /Users/sbhandari/Documents/GitHub/epsilon-transformers/models/1layer_0.15_0.6_lr0.02_2000_thry2/epsilon-transformers/models/linear_mess3_thry2/1layer_0.15_0.6_lr0.02_2000


In [5]:
from epsilon_transformers.process.processes import PROCESS_REGISTRY
from torch import device
process_name = 'Linear_Mess3'
process_params ={
    "x": 0.15,
    "a": 0.6
}
seq_len = 10
vocab = 3
if process_name in PROCESS_REGISTRY:
    process=PROCESS_REGISTRY[process_name](**process_params)
history=process.generate_process_history(total_length=10)
if torch.cuda.is_available():
    device = device("cuda:0")
elif torch.backends.mps.is_available():
    device = device("mps")
else:
    device = device("cpu")
input_seq=torch.tensor([history.symbols],dtype=torch.long,device=device)    

In [4]:
def measure_residual_norms(model, input_seq,layer_idx=None):
    def filter(name):
        return any(name.endswith(suffix) for suffix in ['out','_normalized','_pre','_post','_mid'])
    with torch.no_grad():
        _,cache=model.run_with_cache(input_seq,names_filter=filter)
    norms=[]
    for hook_name, act in cache.items():
        if act.dim()>=2:
            norm_val=act.flatten(start_dim=2).norm(dim=-1).mean().item()
        else:
            norm_val=act.norm().item()
        match=re.search(r'blocks\.(\d+)\.', hook_name)
        layer_idx=int(match.group(1)) if match else -1
        component=hook_name.split('.')[-1]
        norms.append({
            "layer_idx": layer_idx,
            "hook_name": hook_name,
            "component": component,
            "norm": norm_val  })
    return pd.DataFrame(norms)

In [6]:
import torch
import numpy as np
from sklearn.linear_model import LinearRegression

def analyze_mlp_math(model, process, layer_idx=0, num_seqs=128, seq_len=10, device=None):
    """
    Check whether the MLP at layer `layer_idx` implements a token‑dependent
    linear map:  mlp_out ≈ M_z * mlp_in for each token z in {0,1,2}.
    """
    if device is None:
        if torch.backends.mps.is_available():
            device = torch.device("mps")
        else:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Generate a batch of sequences: [B, T]
    if hasattr(process, "generate_batch_gpu"):
        input_batch = process.generate_batch_gpu(
            batch_size=num_seqs, seq_len=seq_len, device=device
        )
    else:
        seqs = [
            process.generate_process_history(total_length=seq_len).symbols
            for _ in range(num_seqs)
        ]
        input_batch = torch.tensor(seqs, dtype=torch.long, device=device)

    # 2. Run model, cache LN2 (MLP input) and MLP out for that layer
    cache_names = [
        f"blocks.{layer_idx}.ln2.hook_normalized",
        f"blocks.{layer_idx}.hook_mlp_out",
    ]
    with torch.no_grad():
        _, cache = model.run_with_cache(input_batch, names_filter=cache_names)

    # Shapes: [B, T, d_model]
    mlp_in = cache[cache_names[0]].cpu().numpy()
    mlp_out = cache[cache_names[1]].cpu().numpy()
    tokens = input_batch.cpu().numpy()  # [B, T]

    # Flatten batch + time: [B*T, ...]
    B, T, D = mlp_in.shape
    mlp_in_flat = mlp_in.reshape(B * T, D)
    mlp_out_flat = mlp_out.reshape(B * T, D)
    tokens_flat = tokens.reshape(B * T)

    results = {}

    for token_id in range(3):  # tokens 0,1,2 for Mess3
        mask = (tokens_flat == token_id)
        X = mlp_in_flat[mask]
        Y = mlp_out_flat[mask]

        if X.shape[0] < D:  # need enough samples
            print(f"Token {token_id}: not enough samples ({X.shape[0]}), skipping.")
            continue

        # Fit linear map Y ≈ X @ M  (no bias; LN should remove mean shifts)
        reg = LinearRegression(fit_intercept=False).fit(X, Y)
        M_learned = reg.coef_        # [D, D]
        r2 = reg.score(X, Y)

        results[token_id] = {"M": M_learned, "R2": r2}
        print(f"Token {token_id}: MLP linear fit R^2 = {r2:.4f}")

    return results

In [7]:
res = analyze_mlp_math(model, process, layer_idx=0, num_seqs=256, seq_len=10)

Token 0: MLP linear fit R^2 = 0.9799
Token 1: MLP linear fit R^2 = 0.9864
Token 2: MLP linear fit R^2 = 0.9698


In [8]:
# in the end we want to do linear regression between the activations and the transformer_input_beliefs
def run_activation_to_beliefs_regression(activations, ground_truth_beliefs):

    # make sure the first two dimensions are the same
    assert activations.shape[0] == ground_truth_beliefs.shape[0]
    assert activations.shape[1] == ground_truth_beliefs.shape[1]

    # flatten the activations
    batch_size, n_ctx, d_model = activations.shape
    belief_dim = ground_truth_beliefs.shape[-1]
    activations_flattened = activations.view(-1, d_model) # [batch * n_ctx, d_model]
    ground_truth_beliefs_flattened = ground_truth_beliefs.view(-1, belief_dim) # [batch * n_ctx, belief_dim]
    
    # run the regression
    regression = LinearRegression()
    regression.fit(activations_flattened, ground_truth_beliefs_flattened)

    # get the belief predictions
    belief_predictions = regression.predict(activations_flattened) # [batch * n_ctx, belief_dim]
    belief_predictions = belief_predictions.reshape(batch_size, n_ctx, belief_dim)

    return regression, belief_predictions



In [9]:
def get_mlp_io_and_beliefs(model, process, regression, layer_idx=0, num_seqs=256, seq_len=10):
    """
    Generates NEW random data, grabs MLP inputs/outputs, and uses the 
    PRE-TRAINED regression probe to estimate beliefs for them.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if torch.backends.mps.is_available(): device = torch.device("mps")

    # 1. Generate NEW random sequences (broad distribution)
    if hasattr(process, "_ensure_gpu_tensors"):
        process._ensure_gpu_tensors(device)
    elif hasattr(process, "transition_matrix"):
         # Manual move if the mixin doesn't have the ensure method public
         if not isinstance(process.transition_matrix, torch.Tensor):
             process.transition_matrix = torch.tensor(process.transition_matrix, dtype=torch.float32)
         process.transition_matrix = process.transition_matrix.to(device)

    if hasattr(process, "generate_batch_gpu"):
        # This should now work because T is on device
        inputs = process.generate_batch_gpu(batch_size=num_seqs, seq_len=seq_len, device=device)
    else:
        seqs = [process.generate_process_history(total_length=seq_len).symbols for _ in range(num_seqs)]
        inputs = torch.tensor(seqs, dtype=torch.long, device=device)
    
    # 2. Run Model
    # We need resid_mid (or whatever layer you trained the probe on) to project to beliefs
    # And MLP input/output for analysis.
    # Note: If you trained probe on Layer 0 resid_post, you should use that. 
    # If analyzing Layer 1 MLP, we usually project Layer 1 resid_mid to beliefs.
    # Let's assume the probe is valid for the layer we are analyzing (or the space is shared).
    cache_names = [
        f"blocks.{layer_idx}.ln2.hook_normalized", # MLP In
        f"blocks.{layer_idx}.hook_mlp_out",         # MLP Out
        f"blocks.{layer_idx}.hook_resid_mid",       # For Belief Projection
    ]
    with torch.no_grad():
        _, cache = model.run_with_cache(inputs, names_filter=cache_names)

    mlp_in = cache[cache_names[0]].cpu().numpy()
    mlp_out = cache[cache_names[1]].cpu().numpy()
    resid_for_probe = cache[cache_names[2]].cpu().numpy()
    tokens = inputs.cpu().numpy()

    # Flatten
    B, T, D = mlp_in.shape
    mlp_in_flat = mlp_in.reshape(B * T, D)
    mlp_out_flat = mlp_out.reshape(B * T, D)
    resid_flat = resid_for_probe.reshape(B * T, D)
    tokens_flat = tokens.reshape(B * T)

    # 3. PROJECT Beliefs using your pre-trained probe
    # regression.predict expects [N, d_model]
    beliefs_flat = regression.predict(resid_flat) # [N, 3]

    return mlp_in_flat, mlp_out_flat, tokens_flat, beliefs_flat


In [10]:
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
import numpy as np

# --- 1. Per-Region Linearity ---
def analyze_per_region_linearity(model, process, regression, layer_idx=1, num_seqs=256, seq_len=32, n_clusters=3):
    print("\n--- 1. Per-Region Linearity Analysis ---")
    mlp_in, mlp_out, tokens, _ = get_mlp_io_and_beliefs(model, process, regression, layer_idx=0, num_seqs=256, seq_len=10)
    
    for token_id in range(3):
        mask = (tokens == token_id)
        X = mlp_in[mask]
        Y = mlp_out[mask]
        
        if len(X) < 50: continue
            
        print(f"Token {token_id}: Global R^2 = {LinearRegression(fit_intercept=False).fit(X, Y).score(X, Y):.4f}")
        
        # Cluster
        kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
        labels = kmeans.fit_predict(X)
        
        for i in range(n_clusters):
            c_mask = (labels == i)
            if c_mask.sum() < 20: continue
            r2 = LinearRegression(fit_intercept=True).fit(X[c_mask], Y[c_mask]).score(X[c_mask], Y[c_mask])
            print(f"  Cluster {i} (n={c_mask.sum()}): R^2 = {r2:.4f}")

# --- 2. Error vs Entropy ---
def analyze_error_entropy(model, process, regression, layer_idx=1, num_seqs=256, seq_len=32):
    print("\n--- 2. Error vs Belief Entropy ---")
    mlp_in, mlp_out, tokens, beliefs = get_mlp_io_and_beliefs(model, process, regression, layer_idx, num_seqs, seq_len)
    
    results = {}
    for token_id in range(3):
        mask = (tokens == token_id)
        X, Y, B = mlp_in[mask], mlp_out[mask], beliefs[mask]
        if len(X) < 50: continue
            
        reg = LinearRegression(fit_intercept=False).fit(X, Y)
        residuals = Y - reg.predict(X)
        e_norm = np.linalg.norm(residuals, axis=1)
        
        # Entropy
        B_safe = np.clip(B, 1e-9, 1.0)
        # Normalize rows to sum to 1 just in case probe output drifted
        B_safe = B_safe / B_safe.sum(axis=1, keepdims=True)
        entropy = -np.sum(B_safe * np.log(B_safe), axis=1)
        
        corr = np.corrcoef(e_norm, entropy)[0, 1]
        print(f"Token {token_id}: Corr(Error, Entropy) = {corr:.4f}")
        results[token_id] = {"reg": reg} # Save for next step

    return results

# --- 3. Orthogonality ---
def analyze_orthogonality(error_results, regression):
    print("\n--- 3. Error Orthogonality ---")
    # P: Belief subspace basis [3, d_model] (from your probe)
    P = regression.coef_ 
    Q, _ = np.linalg.qr(P.T) # Orthonormal basis [d_model, 3]
    
    # We need residuals again. For simplicity, let's re-extract valid data or assume passed.
    # To keep it self-contained, I will assume we are analyzing the *general property* using generic data
    # But usually you pass the residuals from Step 2.
    # Here is a cleaner way:
    # Just inspect the ratio for a fresh batch.
    mlp_in, mlp_out, tokens, _ = get_mlp_io_and_beliefs(model, process, regression)
    
    for token_id in range(3):
        mask = (tokens == token_id)
        if mask.sum() < 50: continue
            
        # Re-fit/Predict to get 'e'
        reg = LinearRegression(fit_intercept=False).fit(mlp_in[mask], mlp_out[mask])
        e = mlp_out[mask] - reg.predict(mlp_in[mask])
        
        # Project e onto belief subspace Q
        e_in_belief = e @ Q @ Q.T
        e_ortho = e - e_in_belief
        
        norm_in = np.linalg.norm(e_in_belief, axis=1).mean()
        norm_ortho = np.linalg.norm(e_ortho, axis=1).mean()
        ratio = norm_in / (norm_in + norm_ortho)
        
        print(f"Token {token_id}: Ratio (Error in Belief Space / Total Error) = {ratio:.4f}")

# --- 4. Matrix Comparison ---
def compare_matrices(model, process, regression, layer_idx=1):
    print("\n--- 4. Learned vs Theoretical Matrix ---")
    P = regression.coef_ # [3, d]
    P_pinv = np.linalg.pinv(P) # [d, 3]
    
    # Get Theoretical S Matrices from Process
    T_theory, _ = process._create_hmm() # [3, 3, 3] (Token, From, To) usually
    
    mlp_in, mlp_out, tokens, _ = get_mlp_io_and_beliefs(model, process, regression, layer_idx)
    
    for token_id in range(3):
        mask = (tokens == token_id)
        if mask.sum() < 50: continue
            
        reg = LinearRegression(fit_intercept=False).fit(mlp_in[mask], mlp_out[mask])
        M_z = reg.coef_ # [d, d]
        
        # Project: P M P+
        M_eff = P @ M_z @ P_pinv
        
        # Theoretical S^z
        # Check process definition. Usually T[token] is the transition matrix.
        # But attention/belief update uses Transpose or Row-stochastic form.
        # Mess3 paper: r_new = r_old * S^z. 
        # If your vectors are row vectors (x @ M), then S^z is T[token].
        # If column vectors (M @ x), it's T[token].T.
        # Standard LinearRegression fits y = x @ A.T + b. So coef_ is A (output x input).
        # Your probe P maps x -> belief.
        # So M_eff maps belief -> belief_update.
        
        S_z = T_theory[token_id] # Adjust transpose if needed based on convention
        
        print(f"\nToken {token_id}:")
        print("Learned (3x3):\n", np.round(M_eff, 3))
        print("Theory (S^z):\n", np.round(S_z, 3))


In [11]:
from epsilon_transformers.analysis.activation_analysis import get_beliefs_for_transformer_inputs
mixed_state_tree = process.derive_mixed_state_presentation(depth=10 + 1)
MSP_transition_matrix = mixed_state_tree.build_msp_transition_matrix()

# in order to plot the belief states in the simplex, we need to get the paths and beliefs from the MSP
tree_paths, tree_beliefs = mixed_state_tree.paths_and_belief_states
msp_beliefs = [tuple(round(b, 5) for b in belief) for belief in tree_beliefs]
print(f"Number of Unique beliefs: {len(set(msp_beliefs))} out of {len(msp_beliefs)}")

# now lets index each belief
msp_belief_index = {b: i for i, b in enumerate(set(msp_beliefs))}
device = 'cpu'
train_config = persister.load_training_config()
transformer_inputs = [x for x in tree_paths if len(x) == 10]
transformer_inputs = torch.tensor(transformer_inputs, dtype=torch.int).to(device)

# print first few batches
print(transformer_inputs[:5])


transformer_input_beliefs, transformer_input_belief_indices = get_beliefs_for_transformer_inputs(transformer_inputs, msp_belief_index, tree_paths, tree_beliefs)
print(f"Transformer Input Beliefs: {transformer_input_beliefs.shape}, Transformer Input Belief Indices: {transformer_input_belief_indices.shape}")

_, activations = model.run_with_cache(transformer_inputs, names_filter=lambda x: 'resid_mid' in x)
#_, activations = model.run_with_cache(transformer_inputs)
#activations['blocks.0.hook_resid_mid'].shape  'ln_final.hook_normalized'
#activations = activations['blocks.3.hook_resid_post']
activations.keys()
print(activations.keys())
#acts = torch.concatenate((activations["blocks.0.ln1.hook_normalized"], activations["blocks.1.ln1.hook_normalized"], activations["blocks.2.ln1.hook_normalized"], activations["blocks.3.ln1.hook_normalized"]), dim=-1)
#acts = activations['ln_final.hook_normalized']
acts = activations['blocks.0.hook_resid_mid']
regression_mid, belief_predictions = run_activation_to_beliefs_regression(acts, transformer_input_beliefs)
print(belief_predictions.shape)

Number of Unique beliefs: 265720 out of 265720
tensor([[0, 0, 0, 2, 2, 0, 0, 0, 1, 0],
        [1, 0, 2, 0, 1, 1, 0, 0, 2, 2],
        [1, 0, 0, 2, 2, 1, 2, 0, 2, 2],
        [0, 0, 0, 2, 2, 0, 0, 0, 1, 1],
        [0, 0, 0, 2, 2, 0, 0, 0, 1, 2]], dtype=torch.int32)
Transformer Input Beliefs: torch.Size([59049, 10, 3]), Transformer Input Belief Indices: torch.Size([59049, 10])
dict_keys(['blocks.0.hook_resid_mid'])
(59049, 10, 3)


In [12]:
_,activations = model.run_with_cache(transformer_inputs, names_filter=lambda x: 'resid_post' in x)
curr_resid_post = activations['blocks.0.hook_resid_post']
print(curr_resid_post.shape)
print(transformer_input_beliefs.shape)
regression_post, belief_predictions_post = run_activation_to_beliefs_regression(curr_resid_post, transformer_input_beliefs)

torch.Size([59049, 10, 6])
torch.Size([59049, 10, 3])


In [13]:
regression, belief_predictions = run_activation_to_beliefs_regression(acts, transformer_input_beliefs)
print(belief_predictions.shape)

(59049, 10, 3)


In [15]:
analyze_per_region_linearity(model, process, regression_mid)


--- 1. Per-Region Linearity Analysis ---
Token 0: Global R^2 = 0.9792
  Cluster 0 (n=355): R^2 = 0.9926
  Cluster 1 (n=304): R^2 = 0.9955
  Cluster 2 (n=203): R^2 = 0.9889
Token 1: Global R^2 = 0.9873
  Cluster 0 (n=577): R^2 = 0.9736
  Cluster 1 (n=92): R^2 = 1.0000
  Cluster 2 (n=222): R^2 = 0.9897
Token 2: Global R^2 = 0.9719
  Cluster 0 (n=187): R^2 = 0.9934
  Cluster 1 (n=83): R^2 = 0.9920
  Cluster 2 (n=537): R^2 = 0.9736


In [16]:
def inspect_variance(model, process, regression, layer_idx=1, num_seqs=256, seq_len=32, n_clusters=3):
    print("\n--- Variance Inspection ---")
    mlp_in, mlp_out, tokens, _ = get_mlp_io_and_beliefs(model, process, regression, layer_idx, num_seqs, seq_len)
    
    for token_id in range(3):
        mask = (tokens == token_id)
        X = mlp_in[mask]
        Y = mlp_out[mask]
        if len(X) < 50: continue

        kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
        labels = kmeans.fit_predict(X)
        
        print(f"\nToken {token_id}:")
        for i in range(n_clusters):
            c_mask = (labels == i)
            Y_c = Y[c_mask]
            if len(Y_c) < 10: continue
            
            # Std Dev of Output Norms
            y_std = Y_c.std(axis=0).mean()
            y_norm = np.linalg.norm(Y_c, axis=1).mean()
            
            # Linearity check again
            reg = LinearRegression(fit_intercept=True).fit(X[c_mask], Y_c)
            r2 = reg.score(X[c_mask], Y_c)
            
            print(f"  Cluster {i}: R2={r2:.4f} | Y_std={y_std:.4f} | Y_norm={y_norm:.4f}")
inspect_variance(model, process, regression, layer_idx=0, num_seqs=256, seq_len=10)


--- Variance Inspection ---

Token 0:
  Cluster 0: R2=0.9952 | Y_std=0.1150 | Y_norm=3.2774
  Cluster 1: R2=0.9932 | Y_std=0.1472 | Y_norm=3.1562
  Cluster 2: R2=0.9882 | Y_std=0.0743 | Y_norm=2.9236

Token 1:
  Cluster 0: R2=0.9933 | Y_std=0.0000 | Y_norm=1.6407
  Cluster 1: R2=0.9898 | Y_std=0.1627 | Y_norm=2.7028
  Cluster 2: R2=0.9756 | Y_std=0.1562 | Y_norm=3.2792

Token 2:
  Cluster 0: R2=0.9771 | Y_std=0.1691 | Y_norm=1.7731
  Cluster 1: R2=1.0000 | Y_std=0.0000 | Y_norm=1.4736
  Cluster 2: R2=0.9812 | Y_std=0.1664 | Y_norm=2.0886


In [18]:
inspect_variance(model, process, regression, layer_idx=0, num_seqs=256, seq_len=10,n_clusters=5)


--- Variance Inspection ---

Token 0:
  Cluster 0: R2=0.9609 | Y_std=0.1965 | Y_norm=12.5761
  Cluster 1: R2=0.9869 | Y_std=0.3172 | Y_norm=13.6696
  Cluster 2: R2=1.0000 | Y_std=0.2531 | Y_norm=14.5618
  Cluster 3: R2=0.9818 | Y_std=0.2914 | Y_norm=13.3801
  Cluster 4: R2=0.9866 | Y_std=0.3058 | Y_norm=13.8296

Token 1:
  Cluster 0: R2=0.9643 | Y_std=0.2115 | Y_norm=12.4563
  Cluster 1: R2=0.9867 | Y_std=0.2727 | Y_norm=13.2021
  Cluster 2: R2=1.0000 | Y_std=0.2605 | Y_norm=13.1953
  Cluster 3: R2=0.9859 | Y_std=0.3006 | Y_norm=13.2518
  Cluster 4: R2=0.9905 | Y_std=0.3401 | Y_norm=13.9356

Token 2:
  Cluster 0: R2=0.9797 | Y_std=0.2608 | Y_norm=12.2855
  Cluster 1: R2=0.9854 | Y_std=0.2840 | Y_norm=13.0214
  Cluster 2: R2=0.9640 | Y_std=0.2495 | Y_norm=13.2656
  Cluster 3: R2=0.9796 | Y_std=0.3161 | Y_norm=12.9355
  Cluster 4: R2=1.0000 | Y_std=0.2543 | Y_norm=12.9621


In [17]:
def analyze_sequence_linear_accuracy(model, process, regression, input_seq, layer_idx=0):
    if input_seq.dim() == 1: input_seq = input_seq.unsqueeze(0)

    # 1. Run Model
    cache_names = [f"blocks.{layer_idx}.hook_mlp_out", f"blocks.{layer_idx}.hook_resid_mid"]
    with torch.no_grad():
        _, cache = model.run_with_cache(input_seq, names_filter=cache_names)

    mlp_out = cache[cache_names[0]][0].cpu().numpy()
    resid_mid = cache[cache_names[1]][0].cpu().numpy()
    tokens = input_seq[0].cpu().numpy()
    
    pred_beliefs = regression.predict(resid_mid)
    
    # Setup Theory
    if hasattr(process, "_create_norm_matrix"):
        S_norm = process._create_norm_matrix()
    else:
        # Re-derive S_norm if needed
        T_raw, _ = process._create_hmm()
        S_norm = np.zeros_like(T_raw)
        for z in range(3):
            S_norm[z] = T_raw[z] / T_raw[z].sum(axis=1, keepdims=True)

    pi = np.array([1/3, 1/3, 1/3])
    curr_lin_belief = pi.copy()
    
    # We also need the INPUT belief to the MLP to check if MLP_out = Input * Matrix
    # We can estimate MLP input belief from resid_mid (since resid_mid -> LN -> MLP)
    # But better to check if MLP_out matches the *change* required.
    
    print(f"\n{'Pos':<3} {'Tok':<3} {'Pred Belief':<25} {'Theory(Lin)':<25} {'L2 Error':<8}")
    print("-" * 75)

    for d, token in enumerate(tokens):
        token_val = int(token)
        
        # Theory Update
        curr_lin_belief = curr_lin_belief @ S_norm[token_val]
        
        # Model Prediction
        pred = pred_beliefs[d]
        
        # Error
        error = np.linalg.norm(pred - curr_lin_belief)
        
        def fmt(v): return f"[{v[0]:.2f} {v[1]:.2f} {v[2]:.2f}]"
        
        print(f"{d:<3} {token_val:<3} {fmt(pred):<25} {fmt(curr_lin_belief):<25} {error:<8.4f}")

# Usage:
# analyze_sequence_linear_accuracy(model, process, regression, input_seq)


In [18]:
def analyze_linear_mechanism_full(model, process, regression_mid, regression_post, input_seq, layer_idx=0):
    if input_seq.dim() == 1: input_seq = input_seq.unsqueeze(0)

    # 1. Run Model - Get Mid (Input) and Post (Output)
    cache_names = [
        f"blocks.{layer_idx}.hook_resid_mid", 
        f"blocks.{layer_idx}.hook_resid_post",
        f"blocks.{layer_idx}.hook_mlp_out"
    ]
    with torch.no_grad():
        _, cache = model.run_with_cache(input_seq, names_filter=cache_names)

    resid_mid = cache[cache_names[0]][0].cpu().numpy()
    resid_post = cache[cache_names[1]][0].cpu().numpy()
    mlp_out = cache[cache_names[2]][0].cpu().numpy()
    tokens = input_seq[0].cpu().numpy()
    
    # 2. Project Both to Belief Space
    pred_con = regression_mid.predict(resid_mid)   # Model's Constrained Belief
    pred_lin = regression_post.predict(resid_post)  # Model's Final Belief (after MLP)
    
    # 3. Setup Theory
    T_raw, _ = process._create_hmm() 
    T_decay = T_raw.sum(axis=0) 
    
    if hasattr(process, "_create_norm_matrix"):
        S_norm = process._create_norm_matrix()
    else:
        S_norm = np.zeros_like(T_raw)
        for z in range(3):
            S_norm[z] = T_raw[z] / T_raw[z].sum(axis=1, keepdims=True)

    pi = np.array([1/3, 1/3, 1/3])
    curr_lin_belief = pi.copy()
    
    def fmt(v): return f"[{v[0]:.2f} {v[1]:.2f} {v[2]:.2f}]"
    
    print(f"\n{'Pos':<3} {'Tok':<3} {'Err(In)':<8} {'Err(Out)':<8} {'MLP_Norm':<8} {'Theory(Con)':<20} {'Pred(Con)':<20} {'Theory(Lin)':<20} {'Pred(Lin)':<20}")
    print("-" * 140)

    for d, token in enumerate(tokens):
        token_val = int(token)
        pos = d + 1
        
        # A. Theory Linear
        S_z = S_norm[token_val]
        curr_lin_belief = curr_lin_belief @ S_z
        
        # B. Theory Constrained
        curr_con_belief = pi.copy()
        for i in range(pos):
            z_i = int(tokens[i])
            S_zi = S_norm[z_i]
            power = (d - i)
            T_pow = np.linalg.matrix_power(T_decay, power)
            term = (pi @ S_zi @ T_pow) - pi
            curr_con_belief += term
            
        # C. Comparison
        # Input Error: Does resid_mid match Theory(Con)?
        err_in = np.linalg.norm(pred_con[d] - curr_con_belief)
        
        # Output Error: Does resid_post match Theory(Lin)?
        err_out = np.linalg.norm(pred_lin[d] - curr_lin_belief)
        
        # MLP Activity
        mlp_norm = np.linalg.norm(mlp_out[d])
        
        print(f"{d:<3} {token_val:<3} {err_in:<8.4f} {err_out:<8.4f} {mlp_norm:<8.2f} {fmt(curr_con_belief):<20} {fmt(pred_con[d]):<20} {fmt(curr_lin_belief):<20} {fmt(pred_lin[d]):<20}")

# Usage:
# analyze_linear_mechanism_full(model, process, regression, input_seq)


In [21]:
# Usage:
history=process.generate_process_history(total_length=10)
input_seq=torch.tensor([history.symbols],dtype=torch.long,device=device)
input_seq=torch.tensor([1,0,1,1,1,2,1,0,2,1],dtype=torch.long,device=device)
# analyze_sequence_linear_accuracy(model, process, regression_mid=regression_mid,regression_post=regression_post, input_seq=input_seq)

In [22]:
analyze_linear_mechanism_full(model, process, regression_mid=regression_mid, regression_post=regression_post, input_seq=input_seq)


Pos Tok Err(In)  Err(Out) MLP_Norm Theory(Con)          Pred(Con)            Theory(Lin)          Pred(Lin)           
--------------------------------------------------------------------------------------------------------------------------------------------
0   1   0.1154   0.0396   1.64     [0.24 0.52 0.24]     [0.16 0.61 0.23]     [0.24 0.52 0.24]     [0.25 0.49 0.26]    
1   0   0.0418   0.0473   3.44     [0.47 0.34 0.19]     [0.44 0.35 0.21]     [0.47 0.32 0.20]     [0.43 0.35 0.22]    
2   1   0.0951   0.0274   3.43     [0.31 0.53 0.16]     [0.37 0.45 0.18]     [0.30 0.52 0.18]     [0.30 0.50 0.20]    
3   1   0.0685   0.0149   3.10     [0.23 0.63 0.14]     [0.25 0.57 0.17]     [0.21 0.62 0.17]     [0.22 0.61 0.17]    
4   1   0.1129   0.0324   3.42     [0.18 0.69 0.13]     [0.25 0.60 0.15]     [0.17 0.67 0.15]     [0.19 0.65 0.16]    
5   2   0.1052   0.0355   1.60     [0.15 0.43 0.41]     [0.23 0.36 0.41]     [0.18 0.39 0.43]     [0.21 0.39 0.41]    
6   1   0.0489   0.0200  

In [8]:
def check_embedding_resid_diff(model, process, layer_idx=0):
    """
    Compares the raw Token Embedding (+Pos Embed) with resid_mid at Position 0.
    This checks if 'resid_mid' is just the embedding or if it has been transformed.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if torch.backends.mps.is_available(): device = torch.device("mps")
    model.to(device)
    
    if hasattr(process, "_ensure_gpu_tensors"):
        process._ensure_gpu_tensors(device)
    elif hasattr(process, "transition_matrix"):
         # Manual move if accessible
         if isinstance(process.transition_matrix, np.ndarray):
             process.transition_matrix = torch.tensor(process.transition_matrix, dtype=torch.float32)
         process.transition_matrix = process.transition_matrix.to(device)
         # Also check if 'T' is a separate attribute
         if hasattr(process, "T") and isinstance(process.T, torch.Tensor):
             process.T = process.T.to(device)

    # 1. Generate Batch
    if hasattr(process, "generate_batch_gpu"):
        inputs = process.generate_batch_gpu(batch_size=1, seq_len=1, device=device)
    # else:
    #     hist = process.generate_process_history(total_length=10)
    #     inputs = torch.tensor([hist.symbols], dtype=torch.long, device=device).repeat(128, 1)

    # 2. Run Model
    cache_names =[f"blocks.{layer_idx}.hook_resid_mid",
                  f"blocks.{layer_idx}.ln1.hook_normalized",
                  f"blocks.{layer_idx}.hook_resid_pre",
                  f"blocks.{layer_idx}.hook_attn_out"]
    with torch.no_grad():
        _, cache = model.run_with_cache(inputs, names_filter=cache_names)
    
    # Resid Mid at Pos 0: [Batch, 1, d_model]
    r_mid_0 = cache[cache_names[0]][:, 0, :]
    r_resid_pre_0=cache[cache_names[2]][:,0,:]
    ln1_normalized_0=cache[cache_names[1]][:,0,:]
    attn_out_0=cache[cache_names[3]][:,0,:]
    diff1=attn_out_0-ln1_normalized_0
    diff1_norms=diff1.norm(dim=1)
    print(f"Mean Norm of (Attn_Out - LN1_Normalized) at Pos 0: {diff1_norms.mean().item():.6f}")
    diffresid_mid_resid_pre= r_mid_0 - r_resid_pre_0
    diffresid_mid_resid_pre_norms=diffresid_mid_resid_pre.norm(dim=1)
    print(f"Mean Norm of (Resid_Mid - Resid_Pre) at Pos 0: {diffresid_mid_resid_pre_norms.mean().item():.6f}")
    cos_sim=torch.nn.functional.cosine_similarity(ln1_normalized_0, r_resid_pre_0)
    print(f"Cosine Similarity between LN1_Normalized and Resid_Pre at Pos 0: {cos_sim.mean().item():.6f}")
    cos_sim2=torch.nn.functional.cosine_similarity(attn_out_0, r_resid_pre_0)
    print(f"Cosine Similarity between Attn_Out and Resid_Pre at Pos 0: {cos_sim2.mean().item():.6f}")
    
    diff_resid_pre_ln1= r_resid_pre_0 - ln1_normalized_0
    diff_norms_resid_pre_ln1=diff_resid_pre_ln1.norm(dim=1)
    print(f"Mean Norm of (Resid_Pre - LN1_Normalized) at Pos 0: {diff_norms_resid_pre_ln1.mean().item():.6f}")
    
    # 3. Reconstruct 'Pure' Embedding (Token + Pos)
    # W_E: [vocab, d_model]
    # W_pos: [max_pos, d_model]
    tokens = inputs[:, 0] # [Batch]
    
    embeds = model.W_E[tokens] # [Batch, d_model]
    pos_embed = model.W_pos[0] # [d_model] (Pos 0)
    
    # Expected 'Raw' input if no Attention happened
    raw_input = embeds + pos_embed
    
    # 4. Compare
    # Difference = What Attention (and LN if applied before resid_mid) did
    diff = r_mid_0 - raw_input
    print(raw_input)
    print(r_mid_0)
    print(pos_embed)
    print(diff)
    
    # Norms
    diff_norms = diff.norm(dim=1)
    raw_norms = raw_input.norm(dim=1)
    r_mid_norms = r_mid_0.norm(dim=1)
    
    # Cosine Similarity
    cos_sim = torch.nn.functional.cosine_similarity(r_mid_0, raw_input)
    
    print("\n--- Position 0 Analysis: Resid_Mid vs Embedding ---")
    print(f"Mean Raw Embedding Norm: {raw_norms.mean().item():.4f}")
    print(f"Mean Resid_Mid Norm:     {r_mid_norms.mean().item():.4f}")
    print(f"Mean Difference Norm:    {diff_norms.mean().item():.4f}")
    print(f"Mean Cosine Similarity:  {cos_sim.mean().item():.4f}")
    
    if diff_norms.mean().item() < 1e-3:
        print(">> Resid_Mid is identical to Embedding (Attention did nothing).")
    else:
        print(">> Resid_Mid differs from Embedding (Attention/LN modified it).")

# Usage:
# check_embedding_resid_diff(model, process)


In [9]:
check_embedding_resid_diff(model, process)

Moving model to device:  mps
Mean Norm of (Attn_Out - LN1_Normalized) at Pos 0: 5.107202
Mean Norm of (Resid_Mid - Resid_Pre) at Pos 0: 2.394574
Cosine Similarity between LN1_Normalized and Resid_Pre at Pos 0: 0.993982
Cosine Similarity between Attn_Out and Resid_Pre at Pos 0: -0.661625
Mean Norm of (Resid_Pre - LN1_Normalized) at Pos 0: 1.690853
tensor([[-0.5357,  1.2190, -0.2539,  0.1279,  0.1881, -0.5388]],
       device='mps:0', grad_fn=<AddBackward0>)
tensor([[-0.4053,  0.0732,  0.8491,  1.3335, -0.3873,  0.6456]],
       device='mps:0')
tensor([-0.3871,  0.7747, -0.1444, -0.1256,  0.3518, -0.1691], device='mps:0',
       grad_fn=<SelectBackward0>)
tensor([[ 0.1304, -1.1458,  1.1030,  1.2056, -0.5754,  1.1844]],
       device='mps:0', grad_fn=<SubBackward0>)

--- Position 0 Analysis: Resid_Mid vs Embedding ---
Mean Raw Embedding Norm: 1.4763
Mean Resid_Mid Norm:     1.7988
Mean Difference Norm:    2.3946
Mean Cosine Similarity:  -0.0600
>> Resid_Mid differs from Embedding (Attenti